# AI_Agent_Workshop_Day2_03 — Evaluate and Pipeline

In this notebook we evaluate the service assistant and connect the workflow to a lightweight DVC pipeline.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "eval").exists():
    ROOT = Path("day2_workshop")

EVAL_DIR = ROOT / "eval"
DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

eval_df = pd.read_csv(EVAL_DIR / "service_eval_set.csv")
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
eval_df

,question,expected_service_name,expected_jurisdiction_level,expected_responsible_body
0,Who do I contact about garbage pickup?,garbage pickup,Region,Region of Waterloo Waste Management
1,Is childcare a city or provincial service?,child care subsidies,Province,Government of Ontario
2,Who handles my property tax bill in Kitchener?,property tax billing,City,City of Kitchener Revenue Division
3,How do I renew my driver's licence?,drivers licence renewal,Province,Government of Ontario / ServiceOntario
4,Where do I apply for EI?,employment insurance,Federal,Government of Canada
5,I found a pothole on my street. Who fixes it?,road pothole on local street,City,City of Kitchener Transportation Services
6,Who clears snow on regional roads?,snow removal on regional roads,Region,Region of Waterloo Transportation Services
7,Who do I call about restaurant inspections?,public health inspections,Region,Region of Waterloo Public Health


## Evaluation criteria

We will track a small set of task-focused metrics:

- jurisdiction classification accuracy,
- responsible-body match,
- source presence,
- next-step usefulness,
- abstention quality for ambiguous cases.

In [2]:
import re

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_baseline_predict(question: str, df: pd.DataFrame) -> dict:
    q_tokens = set(tokenize(question))
    best_score = -1
    best_row = None
    for _, row in df.iterrows():
        text = " ".join([
            str(row["service_name"]),
            str(row["description"]),
            str(row["keywords"]),
            str(row["responsible_body"]),
        ]).lower()
        score = sum(tok in tokenize(text) for tok in q_tokens)
        if score > best_score:
            best_score = score
            best_row = row
    return {
        "predicted_service_name": best_row["service_name"],
        "predicted_jurisdiction_level": best_row["jurisdiction_level"],
        "predicted_responsible_body": best_row["responsible_body"],
        "sources": [best_row["source_url"]],
    }

In [3]:
predictions = []
for _, row in eval_df.iterrows():
    pred = keyword_baseline_predict(row["question"], catalog)
    pred["question"] = row["question"]
    pred["expected_jurisdiction_level"] = row["expected_jurisdiction_level"]
    pred["expected_responsible_body"] = row["expected_responsible_body"]
    predictions.append(pred)

pred_df = pd.DataFrame(predictions)
pred_df

,predicted_service_name,predicted_jurisdiction_level,predicted_responsible_body,sources,question,expected_jurisdiction_level,expected_responsible_body
0,garbage pickup,Region,Region of Waterloo Waste Management,[https://www.regionofwaterloo.ca/],Who do I contact about garbage pickup?,Region,Region of Waterloo Waste Management
1,child care subsidies,Province,Government of Ontario,[https://www.ontario.ca/],Is childcare a city or provincial service?,Province,Government of Ontario
2,property tax billing,City,City of Kitchener Revenue Division,[https://www.kitchener.ca/],Who handles my property tax bill in Kitchener?,City,City of Kitchener Revenue Division
3,drivers licence renewal,Province,Government of Ontario / ServiceOntario,[https://www.ontario.ca/],How do I renew my driver's licence?,Province,Government of Ontario / ServiceOntario
4,snow removal on city streets,City,City of Kitchener Operations,[https://www.kitchener.ca/],Where do I apply for EI?,Federal,Government of Canada
5,road pothole on local street,City,City of Kitchener Transportation Services,[https://www.kitchener.ca/],I found a pothole on my street. Who fixes it?,City,City of Kitchener Transportation Services
6,snow removal on regional roads,Region,Region of Waterloo Transportation Services,[https://www.regionofwaterloo.ca/],Who clears snow on regional roads?,Region,Region of Waterloo Transportation Services
7,public health inspections,Region,Region of Waterloo Public Health,[https://www.regionofwaterloo.ca/],Who do I call about restaurant inspections?,Region,Region of Waterloo Public Health


In [4]:
metrics = {
    "jurisdiction_accuracy": float(
        (pred_df["predicted_jurisdiction_level"] == pred_df["expected_jurisdiction_level"]).mean()
    ),
    "responsible_body_accuracy": float(
        (pred_df["predicted_responsible_body"] == pred_df["expected_responsible_body"]).mean()
    ),
    "source_presence_rate": float(
        pred_df["sources"].apply(lambda x: len(x) > 0).mean()
    ),
    "n_examples": int(len(pred_df)),
}
metrics

{'jurisdiction_accuracy': 0.875,
 'responsible_body_accuracy': 0.875,
 'source_presence_rate': 1.0,
 'n_examples': 8}

## Error analysis

In [5]:
errors = pred_df[
    (pred_df["predicted_jurisdiction_level"] != pred_df["expected_jurisdiction_level"]) |
    (pred_df["predicted_responsible_body"] != pred_df["expected_responsible_body"])
]
errors

,predicted_service_name,predicted_jurisdiction_level,predicted_responsible_body,sources,question,expected_jurisdiction_level,expected_responsible_body
4,snow removal on city streets,City,City of Kitchener Operations,[https://www.kitchener.ca/],Where do I apply for EI?,Federal,Government of Canada


In [6]:
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
pred_df.to_json(ARTIFACTS_DIR / "eval_predictions.json", orient="records", indent=2)

print("Saved metrics and predictions to:", ARTIFACTS_DIR.resolve())

Saved metrics and predictions to: D:\Desktop\8010\AI_Agent_Workshop\artifacts


## DVC scaffold

In [7]:
print((ROOT / "dvc.yaml").read_text())

stages:
  prepare_data:
    cmd: python scripts/prepare_data.py
    deps:
      - scripts/prepare_data.py
      - data/service_catalog.csv
    outs:
      - artifacts/service_catalog.cleaned.json

  run_agent_eval:
    cmd: python scripts/run_agent_eval.py
    deps:
      - scripts/run_agent_eval.py
      - eval/service_eval_set.csv
      - artifacts/service_catalog.cleaned.json
    params:
      - retrieval.top_k
      - retrieval.min_score
      - retrieval.keyword_weight
      - retrieval.ambiguity_margin
      - agent.model_name
      - agent.use_tool_calling
      - agent.require_sources
      - agent.default_confidence
      - agent.unclear_confidence
      - evaluation.max_examples
    outs:
      - artifacts/eval_predictions.json
    metrics:
      - artifacts/metrics.json

  report_metrics:
    cmd: python scripts/report_metrics.py
    deps:
      - scripts/report_metrics.py
      - artifacts/metrics.json
    outs:
      - artifacts/metrics_report.md



In [8]:
print((ROOT / "params.yaml").read_text())

retrieval:
  top_k: 3
  min_score: 1
  keyword_weight: 1.0
  ambiguity_margin: 1
agent:
  model_name: gemini-2.5-flash
  use_tool_calling: true
  require_sources: true
  default_confidence: 0.78
  unclear_confidence: 0.35
evaluation:
  max_examples: 100



## Suggested terminal workflow

Run these commands from the project root after installing DVC:

```bash
dvc repro
dvc metrics show
```

## Exercise

1. Add five harder evaluation questions.
2. Compare a keyword baseline, a retrieval-augmented model call, and a tool-using agent.
3. Track the predictions and metrics in the `artifacts/` directory.
4. Update the DVC stages if you add a new retrieval or indexing step.

## Reflection

An agent demo becomes an ML system only when it is grounded, testable, reproducible, and measurable.

## Exercise 1: Add Five Harder Evaluation Questions

In this step, We extend the evaluation dataset with five harder user questions.
These examples contain indirect wording, ambiguity, or realistic service issues.

In [13]:
# Add five harder evaluation questions

harder_questions = pd.DataFrame([
    {
        "question": "My green bin was not collected this morning. Who should I contact?",
        "expected_service_name": "garbage pickup",
        "expected_jurisdiction_level": "Region",
        "expected_responsible_body": "Region of Waterloo Waste Management"
    },
    {
        "question": "I need help understanding my water bill. Which office handles that?",
        "expected_service_name": "water billing",
        "expected_jurisdiction_level": "City",
        "expected_responsible_body": "City of Kitchener Utilities"
    },
    {
        "question": "Who do I contact if my street was not plowed after the snowstorm?",
        "expected_service_name": "snow removal on city streets",
        "expected_jurisdiction_level": "City",
        "expected_responsible_body": "City of Kitchener Operations"
    },
    {
        "question": "Who should I ask about property tax charges on my house?",
        "expected_service_name": "property tax billing",
        "expected_jurisdiction_level": "City",
        "expected_responsible_body": "City of Kitchener Revenue"
    },
    {
        "question": "My recycling boxes were missed this week. Who is responsible?",
        "expected_service_name": "recycling collection",
        "expected_jurisdiction_level": "Region",
        "expected_responsible_body": "Region of Waterloo Waste Management"
    }
])

# Combine the new harder questions with the original evaluation dataset
extended_eval_df = pd.concat([eval_df, harder_questions], ignore_index=True)

# Display the updated evaluation dataset
extended_eval_df

,question,expected_service_name,expected_jurisdiction_level,expected_responsible_body
0,Who do I contact about garbage pickup?,garbage pickup,Region,Region of Waterloo Waste Management
1,Is childcare a city or provincial service?,child care subsidies,Province,Government of Ontario
2,Who handles my property tax bill in Kitchener?,property tax billing,City,City of Kitchener Revenue Division
3,How do I renew my driver's licence?,drivers licence renewal,Province,Government of Ontario / ServiceOntario
4,Where do I apply for EI?,employment insurance,Federal,Government of Canada
5,I found a pothole on my street. Who fixes it?,road pothole on local street,City,City of Kitchener Transportation Services
6,Who clears snow on regional roads?,snow removal on regional roads,Region,Region of Waterloo Transportation Services
7,Who do I call about restaurant inspections?,public health inspections,Region,Region of Waterloo Public Health
8,My green bin was not collected this morning. W...,garbage pickup,Region,Region of Waterloo Waste Management
9,I need help understanding my water bill. Which...,water billing,City,City of Kitchener Utilities


## Exercise 2: Compare Three Approaches

In this step, We compare three approaches on the extended evaluation dataset:

1. keyword baseline  
2. retrieval-augmented model call  
3. tool-using agent  

This helps show how grounding and structured retrieval can improve routing quality.

In [23]:
# Minimal Gemini setup for this notebook

import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

try:
    from google import genai

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

    if GEMINI_API_KEY:
        client = genai.Client(api_key=GEMINI_API_KEY)
        GEMINI_AVAILABLE = True
    else:
        client = None
        GEMINI_AVAILABLE = False

except Exception:
    client = None
    GEMINI_AVAILABLE = False

print("GEMINI_AVAILABLE =", GEMINI_AVAILABLE)

GEMINI_AVAILABLE = True


In [24]:
# Simple retrieval helper based on token overlap

def retrieve_top_k(question: str, df: pd.DataFrame, k: int = 3) -> list[dict]:
    q_tokens = set(tokenize(question))
    scored = []

    for _, row in df.iterrows():
        text = " ".join([
            str(row["service_name"]),
            str(row["description"]),
            str(row["keywords"]),
            str(row["responsible_body"]),
        ]).lower()

        doc_tokens = set(tokenize(text))
        score = len(q_tokens & doc_tokens)

        scored.append({
            "score": score,
            "service_name": row["service_name"],
            "jurisdiction_level": row["jurisdiction_level"],
            "responsible_body": row["responsible_body"],
            "description": row["description"],
            "source_url": row["source_url"],
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:k]

In [26]:
# Retrieval-augmented model prediction
# If Gemini is unavailable or fails, fall back to the top retrieved result.

def rag_model_predict(question: str, df: pd.DataFrame, k: int = 3) -> dict:
    retrieved = retrieve_top_k(question, df, k=k)
    top = retrieved[0]

    if not GEMINI_AVAILABLE:
        return {
            "predicted_service_name": top["service_name"],
            "predicted_jurisdiction_level": top["jurisdiction_level"],
            "predicted_responsible_body": top["responsible_body"],
            "sources": [doc["source_url"] for doc in retrieved if doc["source_url"]],
            "method_note": "fallback retrieval result used because Gemini is unavailable"
        }

    context_text = "\n\n".join([
        f"""Service: {doc['service_name']}
Jurisdiction: {doc['jurisdiction_level']}
Responsible body: {doc['responsible_body']}
Description: {doc['description']}
Source URL: {doc['source_url']}"""
        for doc in retrieved
    ])

    prompt = f"""
You are a public-service routing assistant.

Use only the retrieved context below to answer the question.
Return a JSON object with exactly these keys:
- predicted_service_name
- predicted_jurisdiction_level
- predicted_responsible_body
- sources

Question:
{question}

Retrieved context:
{context_text}
""".strip()

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        import json
        import re

        text = response.text.strip()
        match = re.search(r"\{.*\}", text, re.DOTALL)

        if match:
            parsed = json.loads(match.group(0))
            return {
                "predicted_service_name": parsed.get("predicted_service_name", top["service_name"]),
                "predicted_jurisdiction_level": parsed.get("predicted_jurisdiction_level", top["jurisdiction_level"]),
                "predicted_responsible_body": parsed.get("predicted_responsible_body", top["responsible_body"]),
                "sources": parsed.get("sources", [doc["source_url"] for doc in retrieved if doc["source_url"]]),
                "method_note": "rag model call"
            }

    except Exception as e:
        print("RAG model call failed:", type(e).__name__, e)

    return {
        "predicted_service_name": top["service_name"],
        "predicted_jurisdiction_level": top["jurisdiction_level"],
        "predicted_responsible_body": top["responsible_body"],
        "sources": [doc["source_url"] for doc in retrieved if doc["source_url"]],
        "method_note": "fallback retrieval result after model failure"
    }

In [27]:
# Tool-using agent style predictor
# This simulates a tool workflow:
# 1. retrieve candidate services
# 2. apply a simple routing rule
# 3. return structured output

def tool_agent_predict(question: str, df: pd.DataFrame) -> dict:
    candidates = retrieve_top_k(question, df, k=3)
    chosen = candidates[0]
    lowered = question.lower()

    # Simple routing rule for ambiguous street / snow cases
    if "street" in lowered or "plowed" in lowered or "plow" in lowered:
        city_matches = [c for c in candidates if str(c["jurisdiction_level"]).lower() == "city"]
        if city_matches:
            chosen = city_matches[0]

    return {
        "predicted_service_name": chosen["service_name"],
        "predicted_jurisdiction_level": chosen["jurisdiction_level"],
        "predicted_responsible_body": chosen["responsible_body"],
        "sources": [c["source_url"] for c in candidates if c["source_url"]],
        "method_note": "tool-style retrieval and routing"
    }

In [28]:
# Generic evaluation helper

def evaluate_method(eval_data: pd.DataFrame, predictor_fn, method_name: str) -> tuple[pd.DataFrame, dict]:
    rows = []

    for _, row in eval_data.iterrows():
        pred = predictor_fn(row["question"], catalog)

        rows.append({
            "question": row["question"],
            "expected_service_name": row["expected_service_name"],
            "expected_jurisdiction_level": row["expected_jurisdiction_level"],
            "expected_responsible_body": row["expected_responsible_body"],
            "predicted_service_name": pred.get("predicted_service_name", ""),
            "predicted_jurisdiction_level": pred.get("predicted_jurisdiction_level", ""),
            "predicted_responsible_body": pred.get("predicted_responsible_body", ""),
            "sources": pred.get("sources", []),
            "method_note": pred.get("method_note", "")
        })

    pred_df = pd.DataFrame(rows)

    metrics = {
        "method": method_name,
        "service_name_accuracy": float(
            (pred_df["predicted_service_name"] == pred_df["expected_service_name"]).mean()
        ),
        "jurisdiction_accuracy": float(
            (pred_df["predicted_jurisdiction_level"] == pred_df["expected_jurisdiction_level"]).mean()
        ),
        "responsible_body_accuracy": float(
            (pred_df["predicted_responsible_body"] == pred_df["expected_responsible_body"]).mean()
        ),
        "source_presence_rate": float(
            pred_df["sources"].apply(lambda x: len(x) > 0).mean()
        ),
        "n_examples": int(len(pred_df))
    }

    return pred_df, metrics

In [29]:
# Compare the three approaches

keyword_pred_df, keyword_metrics = evaluate_method(
    extended_eval_df,
    lambda q, df: keyword_baseline_predict(q, df),
    "keyword_baseline"
)

rag_pred_df, rag_metrics = evaluate_method(
    extended_eval_df,
    lambda q, df: rag_model_predict(q, df),
    "retrieval_augmented_model"
)

tool_pred_df, tool_metrics = evaluate_method(
    extended_eval_df,
    lambda q, df: tool_agent_predict(q, df),
    "tool_using_agent"
)

metrics_df = pd.DataFrame([keyword_metrics, rag_metrics, tool_metrics])
metrics_df

RAG model call failed: ClientError 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6.466589316s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model':

,method,service_name_accuracy,jurisdiction_accuracy,responsible_body_accuracy,source_presence_rate,n_examples
0,keyword_baseline,0.769231,0.923077,0.769231,1.0,13
1,retrieval_augmented_model,0.769231,0.923077,0.769231,1.0,13
2,tool_using_agent,0.769231,0.923077,0.769231,1.0,13


### Exercise 2 Result Summary

All three approaches were successfully evaluated on the extended dataset.

The retrieval-augmented model attempted live model calls, but API quota was exceeded during execution, so the fallback retrieval mechanism was used.

This still allowed the comparison to complete successfully and ensured reproducible evaluation results.

## Exercise 3: Track Predictions and Metrics in the `artifacts/` Directory

In this step, I save the predictions and evaluation metrics into the `artifacts/` directory.

This makes the results easier to inspect, reuse, and connect to a simple reproducible pipeline.

In [30]:
# Ensure the artifacts directory exists

from pathlib import Path
import json

ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts directory:", ARTIFACTS_DIR.resolve())

Artifacts directory: D:\Desktop\8010\AI_Agent_Workshop\artifacts


In [31]:
# Save predictions and metrics into the artifacts directory

keyword_pred_df.to_json(
    ARTIFACTS_DIR / "keyword_baseline_predictions.json",
    orient="records",
    indent=2
)

rag_pred_df.to_json(
    ARTIFACTS_DIR / "rag_predictions.json",
    orient="records",
    indent=2
)

tool_pred_df.to_json(
    ARTIFACTS_DIR / "tool_agent_predictions.json",
    orient="records",
    indent=2
)

metrics_df.to_csv(
    ARTIFACTS_DIR / "comparison_metrics.csv",
    index=False
)

with open(ARTIFACTS_DIR / "comparison_metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "keyword_baseline": keyword_metrics,
            "retrieval_augmented_model": rag_metrics,
            "tool_using_agent": tool_metrics
        },
        f,
        indent=2
    )

print("Saved all predictions and metrics to the artifacts directory.")

Saved all predictions and metrics to the artifacts directory.


In [32]:
# List files saved in artifacts/

sorted([p.name for p in ARTIFACTS_DIR.iterdir()])

['.gitkeep',
 'comparison_metrics.csv',
 'comparison_metrics.json',
 'eval_predictions.json',
 'keyword_baseline_predictions.json',
 'metrics.json',
 'metrics_report.md',
 'rag_predictions.json',
 'service_catalog.cleaned.json',
 'service_catalog.cleaned.updated.json',
 'tool_agent_predictions.json']

In [33]:
# Quick preview of saved metrics

metrics_df

,method,service_name_accuracy,jurisdiction_accuracy,responsible_body_accuracy,source_presence_rate,n_examples
0,keyword_baseline,0.769231,0.923077,0.769231,1.0,13
1,retrieval_augmented_model,0.769231,0.923077,0.769231,1.0,13
2,tool_using_agent,0.769231,0.923077,0.769231,1.0,13


## Exercise 4: Update the DVC Stages

In this step, I describe how the DVC pipeline should be updated if a new retrieval or indexing step is added.

Since this notebook now uses retrieval-style evaluation, the pipeline can be extended with a separate indexing stage before evaluation.

### Suggested DVC Stage Update

If I add a new retrieval or indexing step, I should update the DVC pipeline so the indexing process happens before evaluation.

A simple example could look like this:

```yaml
stages:
  build_index:
    cmd: python src/build_index.py
    deps:
      - data/service_catalog.csv
      - src/build_index.py
    outs:
      - artifacts/service_index.json

  evaluate:
    cmd: python src/run_evaluation.py
    deps:
      - data/service_catalog.csv
      - artifacts/service_index.json
      - src/run_evaluation.py
    outs:
      - artifacts/keyword_baseline_predictions.json
      - artifacts/rag_predictions.json
      - artifacts/tool_agent_predictions.json
    metrics:
      - artifacts/comparison_metrics.json



##  Optional short Python print version


```python
# Simple DVC stage update example as text output

dvc_update_example = """
stages:
  build_index:
    cmd: python src/build_index.py
    deps:
      - data/service_catalog.csv
      - src/build_index.py
    outs:
      - artifacts/service_index.json

  evaluate:
    cmd: python src/run_evaluation.py
    deps:
      - data/service_catalog.csv
      - artifacts/service_index.json
      - src/run_evaluation.py
    outs:
      - artifacts/keyword_baseline_predictions.json
      - artifacts/rag_predictions.json
      - artifacts/tool_agent_predictions.json
    metrics:
      - artifacts/comparison_metrics.json
"""

print(dvc_update_example)

### Exercise 4 Result Summary

Because retrieval logic was added in this notebook, the evaluation workflow is no longer just a single-step process.

A better pipeline design is:

1. prepare or build the retrieval index  
2. run evaluation using that index  
3. save predictions and metrics into `artifacts/`

This makes the workflow more modular and easier to reproduce with DVC.

## Reflection

An agent demo becomes a real ML system only when it is grounded, testable, reproducible, and measurable.

In this notebook, grounding comes from using the service catalog and source-based retrieval instead of relying only on free-form model output.

It becomes testable because I created an evaluation dataset, including five harder questions, and compared multiple approaches on the same examples.

It becomes reproducible because the predictions and metrics are saved into the `artifacts/` directory instead of existing only in notebook output.

It becomes measurable because I tracked evaluation metrics such as service name accuracy, jurisdiction accuracy, responsible body accuracy, and source presence rate.

Overall, this exercise helped me understand that building an agent is only the first step.  
To make it useful as an ML system, we also need evaluation, structured outputs, saved artifacts, and pipeline thinking.